# CADENCE — Stage 2 (Gate A) Confirmatory Run

**Frozen protocol.** This notebook runs the pre-registered Stage 2 confirmatory
experiment (10 seeds × 30,000 timesteps × 8 scenarios) on a Colab GPU. Stage 1
already **PASSED** locally (7/8 gates; G8 a transient-load artifact) at commit
`26938d3`, which is the frozen SHA checked out below.

Colab is **only a compute substrate**. Do not edit any config, add/remove seeds,
run a subset of scenarios, or select a checkpoint by test performance — any such
change invalidates the pre-registration
(`docs/gate_a_preregistration.md`, Part 3, rule #3).

**Runtime → Change runtime type → GPU (T4 or A100).** Then run the cells in order.

Expected wall time: ~15–25 h on a free T4 (expect a disconnect — the runner is
resumable, just re-run Cell 5), ~4–10 h on an A100.

## Cell 1 — Clone the repo at the frozen commit

In [ ]:
import subprocess, os

REPO_URL = "https://github.com/lalith557/CADENCE.git"
# Frozen SHA that produced R-Gate-A-stage1-pass locally. Do NOT use HEAD.
FROZEN_COMMIT = "26938d3bd57ac55b1daeeb940f8c1a61694594c3"

subprocess.check_call(["git", "clone", "--quiet", REPO_URL, "/content/cadence"])
os.chdir("/content/cadence")
subprocess.check_call(["git", "checkout", "--quiet", FROZEN_COMMIT])
print("HEAD =", subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip())

## Cell 2 — Install dependencies

If this Colab's PyTorch / SB3 pins differ from the local `.venv`, the sanity
check in Cell 4 will catch it — match versions before running Stage 2 if so.

In [ ]:
!pip install -q -e /content/cadence
!pip install -q fastapi httpx kaggle  # api tests + dataset download
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## Cell 3 — Fetch the Credit Card Fraud dataset

The dataset is Kaggle-licensed and is **not** committed to the repo. Add a Colab
secret named `KAGGLE_JSON` (left sidebar → 🔑) whose value is the raw contents of
your `kaggle.json` API token, then run this cell.

In [ ]:
from google.colab import userdata
import os, pathlib

kaggle_json = userdata.get("KAGGLE_JSON")
pathlib.Path("/root/.kaggle").mkdir(exist_ok=True)
pathlib.Path("/root/.kaggle/kaggle.json").write_text(kaggle_json)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d mlg-ulb/creditcardfraud -p /content/cadence/Dataset/ --unzip
!mkdir -p "/content/cadence/Dataset/Credit Card Fraud Detection"
!mv "/content/cadence/Dataset/creditcard.csv" "/content/cadence/Dataset/Credit Card Fraud Detection/creditcard.csv"
!ls -la "/content/cadence/Dataset/Credit Card Fraud Detection/"

## Cell 4 — Sanity check (must PASS before Stage 2)

If any test fails: **STOP.** The frozen protocol is not intact on this instance
(usually a dependency-version drift). Fix `pip` pins to match the local `.venv`
before running Stage 2.

In [ ]:
!cd /content/cadence && python -m pytest tests/unit/test_rso_audit.py tests/unit/test_w38_per_seed_training.py tests/unit/test_w39_dual_lr_calmed.py tests/unit/test_w40_reward_component_callback.py tests/unit/test_run_experiment.py -q

## Cell 5 — Run Stage 2 (resumable)

Runs 10 seeds × 30k timesteps × 8 scenarios per the pre-registration. Colab may
disconnect mid-run — the runner's atomic ledger + per-seed checkpoints survive it.
**Re-running this exact cell resumes from the last COMPLETED seed.**

In [ ]:
!cd /content/cadence && python run_experiment.py --config configs/gate_a.yaml --stage 2 --resume

## Cell 6 — Monitor progress (optional)

Run this in a separate cell every ~2 h while Stage 2 is going. It reports gate
diagnostics so far and the per-seed status in the ledger.

In [ ]:
!cd /content/cadence && python scripts/evaluate_stage1_gates.py --log logs/stage2.log 2>/dev/null || true
import json, pathlib
led = pathlib.Path("/content/cadence/experiments/gate_a_ledger.json")
if led.exists():
    d = json.loads(led.read_text())
    seeds = d.get("stages", {}).get("2", {}).get("seeds", [])
    print(json.dumps([{"seed": s["seed"], "status": s["status"], "wall_s": s.get("wall_seconds")} for s in seeds], indent=2))
else:
    print("ledger not created yet")

## Cell 7 — Bundle & download results

Run once **all 10 seeds show `COMPLETED`** in Cell 6. Bundles just the results
(ledger, per-seed artifacts, MLflow DB, log) and downloads them.

In [ ]:
!cd /content/cadence && tar czf /content/gate_a_stage2_results.tar.gz experiments/gate_a_ledger.json experiments/gate_a_stage2/ experiments/rso_ppo_phase_a_seed*.zip experiments/rso_ppo_phase_a_seed*.vecnorm.pkl experiments/mlflow.db logs/stage2.log

from google.colab import files
files.download("/content/gate_a_stage2_results.tar.gz")

## After download — on the local machine

```bash
mv experiments/mlflow.db experiments/mlflow.db.bak-pre-colab-import
tar xzf ~/Downloads/gate_a_stage2_results.tar.gz -C .
python scripts/evaluate_stage1_gates.py     # sanity-check imported state
# Compile R-Gate-A-final in docs/results.md from experiments/gate_a_stage2/seed_*.json
# against the four pre-registered verdict criteria.
```

### Not allowed on Colab (per pre-registration)

- Modifying `configs/gate_a.yaml`
- Modifying `cadence/rso/*`, `benchmarks/phase_a_run.py`, or `run_experiment.py`
  from the frozen commit
- Adding/removing seeds, or running a subset of scenarios
- Selecting a checkpoint by test-set performance

### Provenance to record in `R-Gate-A-final`

- The exact `FROZEN_COMMIT` SHA used on Colab
- The Colab instance type (T4 / A100)
- Per-seed and aggregate wall time
- Confirmation the full ledger + MLflow DB were imported back to local